# CQL policy evaluation on 30 unseen MS MARCO queries

This notebook evaluates the trained `cql_policy.pt` on the first 30 queries from `data/topics.ms-marco-dev.tsv`, using relevance judgments from `data/qrels.ms-marco-dev.tsv`. It records per-step NDCG@50 gain and action cost.

In [4]:
from __future__ import annotations
import ast, sys
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, torch
from d3rlpy.algos import DiscreteCQLConfig
from d3rlpy.constants import ActionSpace
from d3rlpy.dataset import MDPDataset
from d3rlpy.models.encoders import VectorEncoderFactory
from sentence_transformers import SentenceTransformer
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.agents.prf import PRFAgent
from src.agents.reformulate import ReformulationAgent
from src.agents.rerank import RerankingAgent
from src.simulation import ACTION_NAMES, ACTION_STOP, SimConfig, Simulation
from src.utils.retriever import Retriever, create_retriever_callable
TOPICS_PATH = ROOT / 'data/topics.ms-marco-dev.tsv'
QRELS_PATH = ROOT / 'data/qrels.ms-marco-dev.tsv'
TRAJECTORIES_PATH = ROOT / 'data/trajectories_expert_5k.csv'
MODEL_PATH = ROOT / 'cql_policy.pt'
RESULTS_PATH = ROOT / 'eval_cql_30_unseen_queries.csv'
MAX_STEPS, TOP_K, NDCG_K = 3, 50, 50

In [5]:
topics = pd.read_csv(TOPICS_PATH, sep='\t', dtype={'query_id': str}).drop_duplicates('query_id').head(30).reset_index(drop=True)
judgments = pd.read_csv(QRELS_PATH, sep='\t', dtype={'query_id': str, 'doc_id': str})
qrels = defaultdict(dict)
for row in judgments.itertuples(index=False): qrels[str(row.query_id)][str(row.doc_id)] = int(row.relevance)
qrels = dict(qrels)
if len(topics) != 30: raise ValueError(f'Expected 30 topics, found {len(topics)}')
topics = topics[topics.query_id.isin(qrels)].reset_index(drop=True)
if len(topics) != 30: raise ValueError('Some selected topics have no qrels')
training = pd.read_csv(TRAJECTORIES_PATH)
states = np.vstack(training.state.map(ast.literal_eval).to_numpy()).astype(np.float32)
actions = training.action.to_numpy(np.int64); rewards = training.reward.to_numpy(np.float32)
terminals = training.terminal.to_numpy(np.float32); timeouts = training.timeout.to_numpy(np.float32)
dataset = MDPDataset(states, actions, rewards, terminals, timeouts, action_space=ActionSpace.DISCRETE, action_size=int(actions.max()+1))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
policy = DiscreteCQLConfig(batch_size=256, gamma=.99, learning_rate=3e-4, encoder_factory=VectorEncoderFactory(hidden_units=[256,128]), n_critics=2, target_update_interval=8000, alpha=1.0).create(device=device)
policy.build_with_dataset(dataset); policy.load_model(str(MODEL_PATH))
encoder = SentenceTransformer('all-MiniLM-L6-v2')
retriever = create_retriever_callable(Retriever())
agents = [ReformulationAgent(encoder), RerankingAgent(encoder), PRFAgent(encoder, num_expansion_terms=5)]
sim = Simulation(encoder, retriever, agents, SimConfig(max_steps=MAX_STEPS, top_k_rerank=TOP_K, ndcg_k=NDCG_K))
print(f'{len(topics)} selected queries ready; device={device}; state dimension={states.shape[1]}')

2026-09-22 12:59.30 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(399,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1888.95it/s]


30 selected queries ready; device=cuda:0; state dimension=399


In [6]:
def evaluate(query_id, query_text):
    ids, scores, corpus = retriever(query_text, TOP_K)
    original = encoder.encode(query_text, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    query, previous, embedding = query_text, None, original.copy(); last = [False, False, False]; total_cost = total_latency = 0.0; rows = []
    for step in range(MAX_STEPS):
        state = sim.build_state(query, ids, scores, step, last, previous, original, total_latency, total_cost, len(query.split()), embedding)
        before = sim.compute_ndcg(ids, qrels[query_id], NDCG_K); valid = sim._valid_actions_mask(last)
        raw = int(np.asarray(policy.predict(state[None])).reshape(-1)[0]); action = raw if valid[raw] else ACTION_STOP
        new_query, new_ids, new_scores, metrics, latency, cost = sim.compute_effects(action, query, ids, scores, qrels[query_id], corpus_data=corpus)
        cost = float(cost or 0.0); after = float(metrics['ndcg']); total_cost += cost; total_latency += float(latency)
        rows.append({'query_id': query_id, 'query_text': query_text, 'step': step, 'raw_action': raw, 'raw_action_name': ACTION_NAMES[raw], 'action': action, 'action_name': ACTION_NAMES[action], 'mask_override': raw != action, 'ndcg_before': before, 'ndcg_after': after, 'ndcg_gain': after-before, 'cost': cost, 'cumulative_cost': total_cost, 'latency_ms': float(latency)})
        if action == ACTION_STOP: break
        previous, query, ids, scores = list(ids), new_query, list(new_ids), np.asarray(new_scores, dtype=np.float32)
        embedding = encoder.encode(query, convert_to_numpy=True, show_progress_bar=False).astype(np.float32); last = [action == 0, action == 1, action == 2]
    return rows
records, errors = [], []
for row in topics.itertuples(index=False):
    try: records.extend(evaluate(str(row.query_id), row.query_text))
    except Exception as exc: errors.append({'query_id': str(row.query_id), 'error': repr(exc)})
results = pd.DataFrame(records)
if results.empty: raise RuntimeError('No query was evaluated successfully')
results.to_csv(RESULTS_PATH, index=False)
summary = results.groupby('query_id').agg(steps=('step','count'), final_ndcg=('ndcg_after','last'), total_ndcg_gain=('ndcg_gain','sum'), total_cost=('cost','sum')).reset_index()
print(summary[['final_ndcg','total_ndcg_gain','total_cost','steps']].mean())
print(results.action_name.value_counts())
print(f'queries={len(summary)}, failures={len(errors)}, saved={RESULTS_PATH}')

Error retrieving documents: HTTPSConnectionPool(host='opensearch.pads.fim.uni-passau.de', port=443): Max retries exceeded with url: /msmarco-v2.1-segmented/_search (Caused by NameResolutionError("HTTPSConnection(host='opensearch.pads.fim.uni-passau.de', port=443): Failed to resolve 'opensearch.pads.fim.uni-passau.de' ([Errno 11001] getaddrinfo failed)"))
Error retrieving documents: HTTPSConnectionPool(host='opensearch.pads.fim.uni-passau.de', port=443): Max retries exceeded with url: /msmarco-v2.1-segmented/_search (Caused by NameResolutionError("HTTPSConnection(host='opensearch.pads.fim.uni-passau.de', port=443): Failed to resolve 'opensearch.pads.fim.uni-passau.de' ([Errno 11001] getaddrinfo failed)"))
Error retrieving documents: HTTPSConnectionPool(host='opensearch.pads.fim.uni-passau.de', port=443): Max retries exceeded with url: /msmarco-v2.1-segmented/_search (Caused by NameResolutionError("HTTPSConnection(host='opensearch.pads.fim.uni-passau.de', port=443): Failed to resolve 'op

RuntimeError: No query was evaluated successfully